## **Règles de nettoyage**

### **1. raw_campaign_spend_export**

Nous avons 3 cas différents.

**CAS A : doublon exact**  

Si toutes les colonnes sont identiques, on pourra garder une seule ligne

**CAS B : même nom de campagne + même période (debut et fin), mais données différentes**

Par exemple :   
campaign_name : promo ramadan bissap  
date_start : 2026-04-01  
date_end : 2026-04-29

Mais : 
objective : CONVERSIONS  
impressions : 53 488  
clicks : 834 clicks

objective : TRAFFIC  
impressions : 226 620  
clicks : 2 565 clicks  

Ce ne sont clairement pas des doublons exacts : les métriques et l'objectif diffèrent.  
Donc on supprimera uniquement lorsque toutes les valeurs des colonnes sont identiques





#### **Normalisation des objectifs**

Nous avons actuellement :

- Trafic 
- Traffic
- trafic
- Engagement
- engagement
- CONVERSIONS
- Conversions
- Notoriété
- Notoriete
- awareness
- conv

Nous allons créer quelque chose comme :  

Engagement / engagement -> engagement  
Notoriété / Notoriete / awareness -> awareness  
CONVERSIONS / Conversions / conv -> conversions  

Mais nous conserverons également l'original dans une colonne du type objective_raw et objective_normalized pour la traçabilité

#### **Dépenses**

Nous avons des formats comme :
- 1 100 000 FCFA
- 114 900 FCFA
- 617700
- 68 900

Il faudra donc transformer spend en spend_fcfa DOUBLE tout en concervant les valeurs de départ (spend_raw)

#### **Impressions / clicks**

Les 6 NULL restent des NULL. On ne fait surtout pas :  
NULL -> 0 car cela changerait le sens de la donnée. Par exemple, il eest normale que Radio et Influenceur n'aient pas de clicks et d'impressions

### **2. raw_media_plan**

#### **Channel**

On va créer :
- FB/IG -> Meta
- Google -> Google Ads ...

#### **Facturation**

On conserve invoiced_fcfa = NULL lorsque l'information manque et on crée : invoiced_missing = TRUE/FALSE  
Les écarts planned vs invoiced restent visibles. Ils feront partie de notre contrôle qualité.

### **3. raw_pos_sales_daily**

#### **Dates manquantes**

Nous avons exactement 14 jours consécutifs sans données (13 -> 26 avril 2026)  
On ne va pas créer artificiellement des ventes à zéro. Dans le staging, nous garderons la période absente et nous pourrons créer un indicateur de qualité **is_missing_sales_day**

Ainsi, quand on calculera les taux d'évolution, nous pourrons éviter de comparer aveuglément une période complète à une période incomplète.

#### **Retour**
on a : 
- 58 lignes avec revenue_fcfa < 0 (-159 829 FCFA)
- 58 lignes avec units_sold < 0 (-120 unités)

On constate aussi qu'il y a des transactions négatives, synonyme d'un retour, nous allons donc conserver les lignes et dériver :
- gross_sales_revenue
- return_revenue
- net_revenue

Par exemple :

- revenue_fcfa = 30 000, on a vente brute.   
- revenue_fcfa = -2 500, on a retour. 

Le net sera ensuite :
- net_revenue = gross_sales + returns, où les retours restent négatifs.

#### **Communes**

Nous avons :
- Marcory
- MARCORY
- Yopougon
- YOPOUGON

Donc on aura deux nouvelle variable, commune_raw pour l'ancien valeur et commune_normalized pour la nouvelle valeur et on normalise en casse cohérente.

### **4. raw_whatsapp_orders**

Ici nous avons découvert quelque chose de très important.  
order_ref n'est pas une clé fiable   : 

Exemple :

WA-2601-0262 apparaît deux fois avec :

- deux dates différentes (2026-01-01 18:52 et 2026-01-24 14:05) ;
- deux téléphones différents (01 86 23 10 63 et 01 15 45 92 51);
- deux montants différents (1600.0 et 8200.0);
- deux statuts différents (annulé et livrée).
- deux lieux de livraison différentes (riviera3 et angre 8eme tranche)

Donc, on ne va pas faire drop_duplicates(order_ref) mais nous allons plutôt créer un identifiant technique (whatsapp_row_id) et conserver order_ref comme référence métier non fiable.  
Ensuite, éventuellement une variable order_ref_repeated = TRUE pour signaler les références réutilisées.

#### **Statuts**

Nous allons normaliser :
- livré, Livré, LIVRE, livrée vers delivered
- annulé, annulée vers cancelled
- en cours vers pending

#### **Montants**

Pour le moment les 131 valeurs manquantes resteront manquantes

#### **Téléphones**

On créera phone_raw et customer_phone_normalized, par exemple :
- 05 48 68 72 57 -> +2250548687257

### **5. raw_social_comments**

Ici les 25 comment_id répétés sont effectivement identiques dans les exemples C000397 et "Abonne toi à ma page ❤️" et présent deux fois à l'identique.  
Donc ici, un doublon exact pourra être supprimé mais nous allons conserver :
- comment_id
- platform
- post_id
- published_at
- author_handle	comment_text
- like_count
- reply_to_id

puis ajouter après l'IA pour :
- trouver le language (language)
- analyser le sentiment du client (sentiment)
- trouver le thème de la discussion (theme)
- trouver le produit dont le commentaire parle (product)
- Savoir si le commentaire est un spam ou nom (is_spam)

Nous sommes maintenant prêt à passer au staging